# 3. Dimensionality reduction: autoencoder benchmark <a id="3"></a>
Experiment / variant: replace the PCA dimensionality-reduction stage with molearn autoencoders (CNN2d, Small FoldingNet, wr2DCNN) on the same fitted CG activation loops used by `07-PCAClusteringVsKinCore`.


## Table of contents

- [0. Paths and config](#0-paths-and-config)
- [1. Baseline: CNN2d_AE](#1-baseline-cnn2d_ae-with-latent_dim--2)
- [2. Baseline: Small FoldingNet AE](#2-baseline-small-foldingnet-ae-with-latent_dimension--2)
- [2b. Baseline: wr2DCNN (writheCH2)](#2b-baseline-wr2dcnn-writhech2-with-latent_dim--2)
- [3. Latent-dimension scan](#3-latent-dimension-scan-cnn2d-vs-small-vs-wr2dcnn)
- [4. Compare scan results](#4-compare-scan-results)


## Backend map

How this notebook connects to `workflow/` modules (arrows point into the notebook; includes transitive `workflow` subdependencies):

![Backend map](images/backend_maps/07b-AutoencoderBenchmark.v2.svg)

<!-- mermaid source (GitHub does not render mermaid in .ipynb; SVG above is for GitHub):
```mermaid
%%{init: {"flowchart": {"nodeSpacing": 12, "rankSpacing": 28, "padding": 4}, "themeVariables": {"fontSize": "11px"}} }%%
flowchart LR
  NB["07b-AutoencoderBenchmark.ipynb"]
  m_ae["replacingPCA/autoencoder_workflow"]
  m_scan["replacingPCA/run_latent_dim_scan"]
  m_export["replacingPCA/ae_aligned_export"]
  m_small["replacingPCA/small_foldingnet_latent"]
  m_wrch2["replacingPCA/wrCNN2D_ch2"]
  m_wr["replacingPCA/wrCNN2D"]
  m_trainer["replacingPCA/wrTrainer"]
  m_util["utilities"]
  m_wr --> m_wrch2
  m_wr --> m_trainer
  m_wrch2 --> m_ae
  m_trainer --> m_ae
  m_small --> m_ae
  m_export --> m_ae
  m_util --> m_ae
  m_ae --> m_scan
  m_export --> m_scan
  m_ae --> NB
  m_scan --> NB
```
-->


## 0. Paths and config <a id="0-paths-and-config"></a>

Point at August `05a` fitted CG loops and a local `Results/cnn2d_fitted/` tree for checkpoints, aligned PDBs, and plots. Run with cwd = repository root (`workflowAugust2026/`), kernel = `molearn_latest`. CUDA is required.


In [ ]:
import os

from workflow.replacingPCA.autoencoder_workflow import AutoencoderWorkflow

DATA_DIR = os.path.abspath("Results/activation_segments/fitted/")
OUT_DIR = os.path.abspath("Results/cnn2d_fitted/")
ATOM_SELECTION = ["CA"]

assert os.path.isdir(DATA_DIR), f"Missing data directory: {DATA_DIR}"
os.makedirs(OUT_DIR, exist_ok=True)
n_pdbs = len(
    [f for f in os.listdir(DATA_DIR) if f.endswith(".pdb") and f != "combined.pdb"]
)
print(f"DATA_DIR = {DATA_DIR}")
print(f"OUT_DIR  = {OUT_DIR}")
print(f"n_pdbs   = {n_pdbs}")

## 1. Baseline: CNN2d_AE with latent_dim = 2 <a id="1-baseline-cnn2d_ae-with-latent_dim--2"></a>

Single training run at the classical 2-D latent bottleneck (closest analogue to plotting PCA on PC1–PC2).

Steps:
1. **prepare_data** — concatenate per-structure PDBs into `combined.pdb`, select CA, standardize.
2. **train_cnn2d_ae** — encode distance matrices → latent → decode coordinates (molearn Trainer MSE).
3. **setup_analysis** — rebuild the train/valid split used during training.
4. **export_kabsch_aligned_datasets** — decode, Kabsch-align onto inputs, write PDBs/NPZ/RMSD CSV.
5. **five metric plots** — RMSD / Rg / RMSF / CA bond / CA angle.


In [ ]:
SUBFOLDER_CNN2D = "cnn2d_ae"

# device=None picks the CUDA GPU with the most free memory.
wf = AutoencoderWorkflow(
    folder_name=DATA_DIR,
    output_base_dir=OUT_DIR,
    manual_seed=25,
    batch_size=8,
    validation_split=0.1,
)
wf.prepare_data(atom_selection=ATOM_SELECTION)
wf.train_cnn2d_ae(
    max_epochs=32,
    patience=32,
    latent_dim=2,
    init_c=32,
    m=2,
    min_size=9,
    output_subfolder=SUBFOLDER_CNN2D,
)

In [ ]:
wf.setup_analysis(atom_selection=ATOM_SELECTION)
aligned_cnn2d = wf.export_kabsch_aligned_datasets()
print("Export keys:", sorted(aligned_cnn2d.keys()))

In [ ]:
wf.plot_rmsd_comparison(aligned_export=aligned_cnn2d, subfolder=SUBFOLDER_CNN2D, show=True)
wf.plot_rg_comparison(aligned_export=aligned_cnn2d, subfolder=SUBFOLDER_CNN2D, show=True)
wf.plot_rmsf_comparison(aligned_export=aligned_cnn2d, subfolder=SUBFOLDER_CNN2D, show=True)
wf.plot_ca_bondlength_comparison(aligned_export=aligned_cnn2d, subfolder=SUBFOLDER_CNN2D, show=True)
wf.plot_ca_angle_comparison(aligned_export=aligned_cnn2d, subfolder=SUBFOLDER_CNN2D, show=True)

## 2. Baseline: Small FoldingNet AE with latent_dimension = 2 <a id="2-baseline-small-foldingnet-ae-with-latent_dimension--2"></a>

Same data and metrics as above, but using the **Small** model from the December BRAF autoencoder workflow (`workflow.train(...)` default). We use a local wrapper (`small_foldingnet_latent.Small_AutoEncoder`) that is equivalent at `latent_dimension=2` and supports larger latent sizes for the scan below.


In [ ]:
SUBFOLDER_SMALL = "small_ae"

wf_small = AutoencoderWorkflow(
    folder_name=DATA_DIR,
    output_base_dir=OUT_DIR,
    manual_seed=25,
    batch_size=8,
    validation_split=0.1,
)
wf_small.prepare_data(atom_selection=ATOM_SELECTION)
wf_small.train_small_ae(
    max_epochs=32,
    patience=32,
    latent_dimension=2,
    output_subfolder=SUBFOLDER_SMALL,
)
wf_small.setup_analysis(atom_selection=ATOM_SELECTION)
aligned_small = wf_small.export_kabsch_aligned_datasets()
wf_small.plot_rmsd_comparison(aligned_export=aligned_small, subfolder=SUBFOLDER_SMALL, show=True)
wf_small.plot_rg_comparison(aligned_export=aligned_small, subfolder=SUBFOLDER_SMALL, show=True)
wf_small.plot_rmsf_comparison(aligned_export=aligned_small, subfolder=SUBFOLDER_SMALL, show=True)
wf_small.plot_ca_bondlength_comparison(aligned_export=aligned_small, subfolder=SUBFOLDER_SMALL, show=True)
wf_small.plot_ca_angle_comparison(aligned_export=aligned_small, subfolder=SUBFOLDER_SMALL, show=True)

## 2b. Baseline: wr2DCNN (writheCH2) with latent_dim = 2 <a id="2b-baseline-wr2dcnn-writhech2-with-latent_dim--2"></a>

Same data and metrics as above, but using the **two-channel writhe + reciprocal-DM** autoencoder from JoshCollab `wr2DCNNAEv2` (`train_writheCH2_ae`). Encoder input is `[W, R=1/D]`; decoder still outputs coordinates. Loss: `L = MSE_offdiag(Ŵ−W) + β·MSE(R̂−R)` with `β=0.0275`.


In [ ]:
SUBFOLDER_WR2D = "writheCH2_ae"

wf_wr2d = AutoencoderWorkflow(
    folder_name=DATA_DIR,
    output_base_dir=OUT_DIR,
    manual_seed=25,
    batch_size=8,
    validation_split=0.1,
)
wf_wr2d.prepare_data(atom_selection=ATOM_SELECTION)
wf_wr2d.train_writheCH2_ae(
    max_epochs=32,
    patience=32,
    latent_dim=2,
    beta=0.0275,
    output_subfolder=SUBFOLDER_WR2D,
)
wf_wr2d.setup_analysis(atom_selection=ATOM_SELECTION)
aligned_wr2d = wf_wr2d.export_kabsch_aligned_datasets()
wf_wr2d.plot_rmsd_comparison(aligned_export=aligned_wr2d, subfolder=SUBFOLDER_WR2D, show=True)
wf_wr2d.plot_rg_comparison(aligned_export=aligned_wr2d, subfolder=SUBFOLDER_WR2D, show=True)
wf_wr2d.plot_rmsf_comparison(aligned_export=aligned_wr2d, subfolder=SUBFOLDER_WR2D, show=True)
wf_wr2d.plot_ca_bondlength_comparison(aligned_export=aligned_wr2d, subfolder=SUBFOLDER_WR2D, show=True)
wf_wr2d.plot_ca_angle_comparison(aligned_export=aligned_wr2d, subfolder=SUBFOLDER_WR2D, show=True)


## 3. Latent-dimension scan (CNN2d vs Small vs wr2DCNN) <a id="3-latent-dimension-scan-cnn2d-vs-small-vs-wr2dcnn"></a>

Full Josh-style scan (option A):

| Setting | Value |
|---|---|
| Latent dims | `4, 6, 10, 25, 36, 54, 64, 81, 120` |
| Repeats | 3 (seeds `25, 26, 27`) |
| Models | `cnn2d_ae`, `small_ae`, `writheCH2_ae` |
| writheCH2 β | `0.0275` |
| Layout | `Results/.../latentDScan/{d}/{model}/repeat_{r}/` |

That is **81** train+eval jobs. Prefer running from a terminal; use `--skip-existing` to resume. To train only the new model against existing CNN2d/Small checkpoints:

```bash
cd replacingPCA
conda activate molearn_latest
python run_latent_dim_scan.py --models writheCH2_ae --skip-existing
# or all three:
python run_latent_dim_scan.py --skip-existing
```

The cell below launches the same full scan in-process. After it finishes (or if you already ran the CLI), section 4 plots the summary CSVs.


In [ ]:
import workflow.replacingPCA.run_latent_dim_scan as lscan

# Full option-A scan (all three models). For writheCH2-only against existing runs:
#   argv = ['--models', 'writheCH2_ae', '--skip-existing', ...]
# For a smoke test, override e.g.:
#   argv = ['--latent-dim', '4', '--repeats', '1', '--models', 'writheCH2_ae', '--skip-existing']
argv = [
    "--data-dir", DATA_DIR,
    "--base-output-dir", OUT_DIR,
    "--latent-scan-root", "latentDScan",
    "--skip-existing",
]
lscan.main(argv)


## 4. Compare scan results <a id="4-compare-scan-results"></a>

Load `latentDScan/comparison/latent_dim_scan_agg_mean_std.csv` (written by the scan) and plot mean ± std validation RMSD versus latent dimension for CNN2d, Small, and wr2DCNN. If the agg file is missing, rebuild summaries from existing run folders with `--summary-only`.


In [ ]:
import glob

import matplotlib.pyplot as plt
import pandas as pd

comp_dir = os.path.join(OUT_DIR, "latentDScan", "comparison")
agg_path = os.path.join(comp_dir, "latent_dim_scan_agg_mean_std.csv")
summary_path = os.path.join(comp_dir, "latent_dim_scan_summary.csv")

if not os.path.isfile(agg_path):
    print(f"Agg CSV missing ({agg_path}); rebuilding with --summary-only ...")
    lscan.main(
        [
            "--data-dir", DATA_DIR,
            "--base-output-dir", OUT_DIR,
            "--latent-scan-root", "latentDScan",
            "--summary-only",
        ]
    )

if os.path.isfile(agg_path):
    agg = pd.read_csv(agg_path)
    display(agg.head())

    fig, ax = plt.subplots(figsize=(7, 4))
    for model, label, color in (
        ("cnn2d_ae", "CNN2d", "C0"),
        ("small_ae", "Small", "C1"),
        ("writheCH2_ae", "wr2DCNN", "C2"),
    ):
        sub = agg[agg["model"] == model].sort_values("latent_dim")
        if sub.empty or "mean_rmsd_valid_mean" not in sub.columns:
            continue
        x = sub["latent_dim"].values
        y = sub["mean_rmsd_valid_mean"].values
        yerr = sub["mean_rmsd_valid_std"].fillna(0).values
        ax.errorbar(x, y, yerr=yerr, marker="o", label=label, color=color, capsize=3)

    ax.set_xlabel("Latent dimension")
    ax.set_ylabel("Mean valid RMSD (Å)")
    ax.set_title("Latent-dim scan: validation RMSD (mean ± std over repeats)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    out_png = os.path.join(comp_dir, "mean_rmsd_valid_vs_latent_dim.png")
    fig.savefig(out_png, dpi=200)
    print(f"Saved {out_png}")
    plt.show()
elif os.path.isfile(summary_path):
    print("Only per-run summary found:")
    display(pd.read_csv(summary_path).head())
else:
    n_ckpts = len(
        glob.glob(os.path.join(OUT_DIR, "latentDScan", "**", "checkpoint_*.ckpt"), recursive=True)
    )
    print(f"No comparison CSVs yet ({n_ckpts} checkpoints under latentDScan/). Run section 3 first.")